In [ ]:
# !pip install -U scikit-learn
import tensorflow as tf

# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

try:
    # %tensorflow_version only exists in Colab.
    %tensorflow_version 2.x
except Exception:
    pass

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

%load_ext tensorboard

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# pip install sympy
# pip install 'h5py==2.10.0' --force-reinstall

## 1- Téléchargez le contenu de la base de données. (fashion_mnist = keras.datasets.fashion_mnist)

In [ ]:
# Keras dispose de fonctions utilitaires pour récupérer et
# charger des jeux de données communs comme Fashion MNIST
# Les données sont divisées en un ensemble de données
# d'entrainement et de données de test
# La variable y étant la variable cible c-à-d la classe
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

## 2- Affichez trois échantillons de cette base de données

In [ ]:
# Affichage de certains échantillons de la base de données
# Ici, on a choisi le 1er, 2ème et 6ème échantillon
from matplotlib import pyplot as plt
plt.subplot(131)
plt.imshow(X_train_full[0])
plt.subplot(132)
plt.imshow(X_train_full[1])
plt.subplot(133)
plt.imshow(X_train_full[5])
plt.show()

## 3- Procédez à une normalisation des données entre 0 et 1

In [ ]:
# Pour entraîner le réseau de neurones avec la descente
# de gradient, on doit mettre à l'échelle les caractéristiques d'entrée.
# On peut normaliser les intensités de pixels à la plage 0-1
# en les divisant par 255 et ce pour simplifier les calculs
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

## 4- Vérifiez la taille de l'échantillon d'entrainement et de test par classe

In [ ]:
# La forme des données
# Les données MNIST se présentent sous la forme d'une matrice de 28*28
# L'ensemble d'entrainement comprend 60000
X_train_full.shape

In [ ]:
# La base de données ne comprend pas de données de validation,
# on peut créer un sous ensemble de validation
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

In [ ]:
X_valid.shape

In [ ]:
X_test.shape

In [ ]:
# La variable cible étant des entiers, on peut renseigner la liste
# des noms de classes si on le désire
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## 5- Développez un RNP à deux couches cachées (de dimensions [300,100]). Utilisez la fonction d'activation LeakyReLU et une initialisation He

In [ ]:
# Instanciation du modèle = Structure séquentielle = Réseau de neurone
# avec des couches en séquence.
# La couche Flatten a pour rôle de convertir chaque image d'entrée en
# un tableau à une dimension. 784 dans notre cas.
# Ensuite, on a la suite des couches cachées et la couche de sortie
# La fonction d'activation est une LeakyReLU et l'initialisation
# est celle de He
tf.random.set_seed(42)
np.random.seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, kernel_initializer="he_normal"),
    keras.layers.LeakyReLU(),
    keras.layers.Dense(100, kernel_initializer="he_normal"),
    keras.layers.LeakyReLU(),
    keras.layers.Dense(10, activation="softmax")
])

## 6- Affichez le résumé du modèle

In [ ]:
# Affichage du résumé du modèle :
model.summary()

# 784 = 28*28
# 235200 = 784 * 300
# 235500 = 235200 + 300 (biais)
# 266610 = 235500 + 30100 + 1010

In [ ]:
# Configuration du réseau de neurone pour l'apprentissage
# La fonction coût utilisée est la cross entropy
# loss=sparse_categorical_crossentropy est utilisée pour la multi-classification.
# Les valeurs prédites sont alors des entiers
# La métrique utilisée est le taux de classification global (metrics=["accuracy"])
# L'optimizer est une descente de gradient et le pas d'apprentissage est 0.001
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=1e-3),
              metrics=["accuracy"])

In [ ]:
# Affichage de la liste des couches du modèle
model.layers

In [ ]:
hidden1 = model.layers[1]
weights = hidden1.get_weights()
weights

## 7- Déterminez le taux de classification global et les taux de classification par classe

In [ ]:
# Le nombre d'époques = 100 (epochs = 100) -->
# la base de données d'entrainement est parcourue au complet 100 fois
history = model.fit(X_train, y_train, epochs=100,
                    validation_data=(X_valid, y_valid))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# La méthode fit() retourne un objet History qui contient les paramètres suivants :
# history.params   : contient les paramètres d'entrainement
# history.epoch    : contient la liste des époques
# history.history  : comprend la perte et la performance mesurées
#                    à la fin de chaque époque sur le jeu d'entraînement et le jeu de validation.
# L'axe vertical varie entre 0 et 1
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()

## 8- Utilisez une normalisation par batch et déterminez le taux de classification global

In [ ]:
# La mise en oeuvre de la normalisation par lots (Batch)
# est simple et intuitive. Il suffit d'ajouter une couche
# BatchNormalization avant ou après la fonction d'activation
# de chaque couche cachée et d'ajouter une couche BN en
# première couche du modèle.
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10, activation="softmax")
])
model.summary()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=1e-3),
              metrics=["accuracy"])

In [ ]:
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

In [ ]:
# Le taux de classification est de ~87%

## 10- Construisez les sous-bases de données X_train_A, X_valid_A et X_test_A.

In [ ]:
# La base de données comprend actuellement 10 classes
# Supposons que la base de données Fashion MNIST ne contienne
# que huit classes : par exemple, toutes les classes à l'exception
# des sandales et des chemises.
print(class_names)

In [ ]:
# On doit donc supprimer les classes 'Sandal', 'Shirt'
# qui correspondent aux classes d'indices 5 et 6
# Et construire les nouvelles bases de données
# X_train_A, X_valid_A et X_test_A
# Attention : les class indices 7, 8, 9 doivent être
# décalés et devenir les nouvelles classes 5, 6, 7
# dans X_train_A, X_valid_A et X_test_A
def split_dataset(X, y):
    y_5_or_6 = (y == 5) | (y == 6)
    y_A = y[~y_5_or_6]
    y_A[y_A > 6] -= 2
    y_B = (y[y_5_or_6] == 6).astype(np.float32)
    return ((X[~y_5_or_6], y_A),
            (X[y_5_or_6], y_B))

(X_train_A, y_train_A), (X_train_B, y_train_B) = split_dataset(X_train, y_train)
(X_valid_A, y_valid_A), (X_valid_B, y_valid_B) = split_dataset(X_valid, y_valid)
(X_test_A, y_test_A),   (X_test_B, y_test_B)   = split_dataset(X_test,  y_test)

X_train_B = X_train_B[:200]
y_train_B = y_train_B[:200]

In [ ]:
X_train_A.shape

In [ ]:
X_train_B.shape

In [ ]:
# Vérification des nouvelles classes de y_train_A
# contenant tous les échantillons sauf 'Sandal', 'Shirt'
y_train_A[:30]

In [ ]:
# Vérification des nouvelles classes de y_train_B
y_train_B[:30]

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

## 11- Développez un modèle A basé sur l'entrainement à partir de ces huit classes en utilisant cinq couches cachées (de dimensions [300,100,50,50,50]), une fonction d'activation selu, une descente de gradient et un taux d'apprentissage = 0,001)

In [ ]:
# Instanciation du modèle A pour la classification
# des 8 classes retenues
# Utilisation de la fonction d'activation selu dans les couches cachées
model_A = keras.models.Sequential()
model_A.add(keras.layers.Flatten(input_shape=[28, 28]))
for n_hidden in (300, 100, 50, 50, 50):
    model_A.add(keras.layers.Dense(n_hidden, activation="selu"))
model_A.add(keras.layers.Dense(8, activation="softmax"))

# Entrainement du modèle A pour la classification
# des 8 classes retenues
model_A.compile(loss="sparse_categorical_crossentropy",
                optimizer=keras.optimizers.SGD(learning_rate=1e-3),
                metrics=["accuracy"])

## 12- Évaluez les performances du modèle A

In [ ]:
history = model_A.fit(X_train_A, y_train_A, epochs=20,
                      validation_data=(X_valid_A, y_valid_A))

In [ ]:
# Sauvegarde du modèle A
model_A.save("my_model_A.h5")

## 13- Développez un modèle B qui considère la classification binaire : la classe sandale = 1 et la classe chemise = 0 en utilisant cinq couches cachées (de dimensions [300,100,50,50,50]), une fonction d'activation selu, une descente de gradient et un taux d'apprentissage = 0,001)

In [ ]:
# Instanciation du modèle B de classification binaire
# Les classes considérées sont (sandals and shirts)
# La fonction d'activation est une sigmoïde dans ce cas.
model_B = keras.models.Sequential()
model_B.add(keras.layers.Flatten(input_shape=[28, 28]))
for n_hidden in (300, 100, 50, 50, 50):
    model_B.add(keras.layers.Dense(n_hidden, activation="selu"))
model_B.add(keras.layers.Dense(1, activation="sigmoid"))
model_B.summary()

## 14- Évaluez les performances du modèle B

In [ ]:
model_B.compile(loss="binary_crossentropy",
                optimizer=keras.optimizers.SGD(learning_rate=1e-3),
                metrics=["accuracy"])

# Entrainement du modèle B
# Test sur les données de valid_B. Le taux de classification
# obtenu est de 97%.
history = model_B.fit(X_train_B, y_train_B, epochs=20,
                      validation_data=(X_valid_B, y_valid_B))

In [ ]:
# Sommaire du modèle B
model_B.summary()

## 15- Utilisez le modèle A pour entrainer le modèle B (apprentissage par transfert). Évaluez les performances de ce modèle

In [ ]:
# Chargement du modèle A
model_A = keras.models.load_model("my_model_A.h5")
model_A.summary()

In [ ]:
# Pour avoir la configuration du modèle
model_A.get_config()

In [ ]:
# Copie du modèle A sur le modèle de B_on_A
# Rappel : le modèle A est celui entrainé sur les 8 classes
# La fonction d'activation est sigmoïde parce que nous avons
# une classification binaire
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation="sigmoid"))

In [ ]:
# Clonage du modèle A avant de réutiliser ses couches pour
# éviter de l'impacter lors de l'entrainement du modèle B
model_A_clone = keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

In [ ]:
# On peut mettre layer.trainable à False pour faire passer tous les poids
# de la couche de "trainable" à "non-trainable".
# Ceci permet de "geler" la couche :
# l'état d'une couche gelée ne sera pas mis à jour pendant l'apprentissage.
# Le taux d'apprentissage est égal à 1e-3
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True

model_B_on_A.compile(loss="binary_crossentropy",
                     optimizer=keras.optimizers.SGD(learning_rate=1e-3),
                     metrics=["accuracy"])

history = model_B_on_A.fit(X_train_B, y_train_B, epochs=16,
                            validation_data=(X_valid_B, y_valid_B))

In [ ]:
# Évaluation du modèle B
model_B.evaluate(X_test_B, y_test_B)

In [ ]:
# Évaluation du modèle B_on_A
model_B_on_A.evaluate(X_test_B, y_test_B)